# ETL Pipeline — Construction du Data Warehouse

**Démarrer PostgreSQL :**
```bash
docker compose -f docker/docker-compose.yml up -d
```

## 2.1 — Connexion & schéma base de données

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
import io
import time
import os
from pathlib import Path
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Chemins
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

# Charger les credentials depuis ../.env
load_dotenv(os.path.sep.join([os.path.dirname(os.getcwd()), '.env']), override=True)
load_dotenv(os.path.sep.join([os.path.dirname(os.getcwd()), 'config', '.env.common']), override=True)

PG_HOST = os.getenv('SCW_DB_HOST')
PG_PORT = os.getenv('SCW_DB_PORT')
PG_DB   = os.getenv('SCW_DB_NAME')
PG_USER = os.getenv('SCW_DB_USER')
PG_PASS = os.getenv('SCW_DB_PASSWORD')

DB_URL = os.getenv('SCW_DB_URL')
engine = create_engine(DB_URL, echo=False, echo_pool=False)

# Helpers
def run_sql(sql, params=None):
    """Exécute une requête SQL sans retour de données."""
    with engine.connect() as con:
        con.execute(text(sql), params or {})
        con.commit()

def query_df(sql):
    """Exécute une requête SQL et retourne un DataFrame."""
    return pd.read_sql(text(sql), engine)

def ingest_df(df, schema, table, method='multi', chunksize=5000):
    """
    Ingestion rapide d'un DataFrame dans PostgreSQL.
    Utilise COPY FROM STDIN (psycopg2) pour les grands datasets (>50k lignes).
    Les colonnes sont explicitement nommées → compatible avec les colonnes SERIAL.
    """
    if len(df) > 50_000:
        # COPY : ~10x plus rapide que INSERT pour les grands volumes
        conn = engine.raw_connection()
        try:
            cur = conn.cursor()
            buf = io.StringIO()
            df.to_csv(buf, index=False, header=False, na_rep='\\N')
            buf.seek(0)
            # ✅ Nommer les colonnes explicitement → PostgreSQL gère le SERIAL tout seul
            cols = ', '.join(df.columns)
            cur.copy_expert(
                f"COPY {schema}.{table} ({cols}) FROM STDIN WITH CSV NULL '\\N'",
                buf
            )
            conn.commit()
        finally:
            cur.close()
            conn.close()
    else:
        df.to_sql(table, engine, schema=schema, if_exists='append',
                  index=False, method=method, chunksize=chunksize)

# Tester la connexion
try:
    with engine.connect() as con:
        version = con.execute(text("SELECT version()")).scalar()
    print(f"✅ PostgreSQL connecté")
    print(f"   {version[:70]}")
    print(f"   Host : {PG_HOST}:{PG_PORT}/{PG_DB}")
except Exception as e:
    print(f"❌ Connexion échouée : {e}")
    print("   → Vérifier que Docker est démarré : docker compose -f docker/docker-compose.yml up -d")

In [ ]:
# 2.1.1 — Création des schémas raw et public (DWH)
run_sql("CREATE SCHEMA IF NOT EXISTS raw")
print("✅ Schémas créés : raw (Data Lake), public (Data Warehouse)")

In [ ]:
# 2.1.2 — Suppression des tables existantes (idempotent)
# CASCADE supprime automatiquement les dépendances FK

TABLES_DWH = [
    'fait_impact_pays_annee',
    'dim_socio_economique',
    'fait_production',
    'fait_impact',
    'dim_produits',
    'dim_temps',
    'dim_pays',
]
TABLES_RAW = [
    'raw.raw_fao_historique',
    'raw.raw_fao_complet',
    'raw.raw_food_production',
    'raw.raw_worldbank',
    'raw.raw_owid_meat',
]

print("🗑️  Suppression des tables existantes...")
for table in TABLES_DWH:
    run_sql(f"DROP TABLE IF EXISTS public.{table} CASCADE")
for table in TABLES_RAW:
    run_sql(f"DROP TABLE IF EXISTS {table} CASCADE")
print("✅ Tables supprimées")

In [ ]:
# 2.1.3 — Création des tables de dimension (DWH)

run_sql("""
CREATE TABLE dim_pays (
    pays_id   SERIAL PRIMARY KEY,
    nom_pays  VARCHAR(100) NOT NULL,
    code_fao  VARCHAR(10),
    code_iso2 VARCHAR(2),
    code_iso3 VARCHAR(3),
    region    VARCHAR(100),
    latitude  DOUBLE PRECISION,
    longitude DOUBLE PRECISION
)""")

run_sql("""
CREATE TABLE dim_produits (
    produit_id     SERIAL PRIMARY KEY,
    nom_fao        VARCHAR(200),
    nom_impact     VARCHAR(200),
    categorie      VARCHAR(100) NOT NULL,
    sous_categorie VARCHAR(100),
    match_quality  VARCHAR(20)
)""")

run_sql("""
CREATE TABLE dim_temps (
    annee_id SERIAL PRIMARY KEY,
    annee    INTEGER NOT NULL,
    decennie INTEGER,
    periode  VARCHAR(10)
)""")

run_sql("""
CREATE TABLE dim_socio_economique (
    socio_id          SERIAL PRIMARY KEY,
    pays_id           INTEGER NOT NULL REFERENCES dim_pays(pays_id),
    annee_id          INTEGER NOT NULL REFERENCES dim_temps(annee_id),
    pib_per_capita    DOUBLE PRECISION,
    taux_urbanisation DOUBLE PRECISION,
    population        BIGINT,
    surface_agricole  DOUBLE PRECISION
)""")

print("✅ Dimensions créées : dim_pays, dim_produits, dim_temps, dim_socio_economique")

In [ ]:
# 2.1.4 — Création des tables de fait (DWH)

run_sql("""
CREATE TABLE fait_production (
    production_id  SERIAL PRIMARY KEY,
    pays_id        INTEGER NOT NULL REFERENCES dim_pays(pays_id),
    produit_id     INTEGER NOT NULL REFERENCES dim_produits(produit_id),
    annee_id       INTEGER NOT NULL REFERENCES dim_temps(annee_id),
    element        VARCHAR(10) NOT NULL,
    quantite_1000t DOUBLE PRECISION
)""")

run_sql("""
CREATE TABLE fait_impact (
    impact_id              SERIAL PRIMARY KEY,
    produit_id             INTEGER NOT NULL REFERENCES dim_produits(produit_id),
    co2_land_use_per_kg    DOUBLE PRECISION,
    co2_animal_feed_per_kg DOUBLE PRECISION,
    co2_farm_per_kg        DOUBLE PRECISION,
    co2_processing_per_kg  DOUBLE PRECISION,
    co2_transport_per_kg   DOUBLE PRECISION,
    co2_packaging_per_kg   DOUBLE PRECISION,
    co2_retail_per_kg      DOUBLE PRECISION,
    co2_total_per_kg       DOUBLE PRECISION,
    freshwater_per_kg      DOUBLE PRECISION,
    scarcity_water_per_kg  DOUBLE PRECISION,
    land_use_per_kg        DOUBLE PRECISION,
    eutrophying_per_kg     DOUBLE PRECISION
)""")

run_sql("""
CREATE TABLE fait_impact_pays_annee (
    impact_pays_id          SERIAL PRIMARY KEY,
    pays_id                 INTEGER NOT NULL REFERENCES dim_pays(pays_id),
    annee_id                INTEGER NOT NULL REFERENCES dim_temps(annee_id),
    produit_id              INTEGER NOT NULL REFERENCES dim_produits(produit_id),
    quantite_1000t          DOUBLE PRECISION,
    quantite_kg             DOUBLE PRECISION,
    co2_total_kg            DOUBLE PRECISION,
    freshwater_total_litres DOUBLE PRECISION,
    land_use_total_m2       DOUBLE PRECISION
)""")

print("✅ Faits créés : fait_production, fait_impact, fait_impact_pays_annee")

In [ ]:
# 2.1.5 — Validation du schéma DWH
print("📋 Tables DWH créées (schéma public) :")
print("=" * 60)

tables = query_df("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
""")

for _, row in tables.iterrows():
    t = row['table_name']
    cols = query_df(f"""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = '{t}'
        ORDER BY ordinal_position
    """)
    print(f"\n  📁 {t} ({len(cols)} colonnes)")
    for _, c in cols.iterrows():
        nullable = '' if c['is_nullable'] == 'YES' else ' NOT NULL'
        print(f"     {c['column_name']:<30} {c['data_type']:<20}{nullable}")

print(f"\n{'='*60}")
print(f"✅ {len(tables)} tables DWH créées")

---
## 2.2 — Data Lake : ingestion des données brutes

Les tables `raw.*` sont un miroir exact des fichiers sources CSV, sans transformation.

In [ ]:
# 2.2.1 — Vérification des fichiers sources
SOURCES = {
    'raw_fao_historique' : DATA_RAW / 'FAO.csv',
    'raw_fao_complet'    : DATA_RAW / 'FAO_complet_1961_2023.csv',
    'raw_food_production': DATA_RAW / 'Food_Production.csv',
    'raw_worldbank'      : DATA_RAW / 'worldbank_socioeco_interp.csv',
    'raw_owid_meat'      : DATA_RAW / 'owid_meat_consumption.csv',
}

print("📂 Vérification des fichiers sources :")
for table, path in SOURCES.items():
    exists = path.exists()
    size   = f"{path.stat().st_size / 1024**2:.1f} MB" if exists else "—"
    status = "✅" if exists else "⚠️  MANQUANT"
    print(f"  {status} {path.name:<45} {size}")

In [ ]:
# 2.2.2 — Ingestion des CSV dans le schéma raw
# Utilise COPY FROM STDIN (psycopg2) pour les grands fichiers → très rapide

# Ensure no left-over transaction is pending
with engine.connect() as con:
    con.rollback()

results = {}

for table_name, csv_path in SOURCES.items():
    if not csv_path.exists():
        print(f"  ⏭️  {table_name} — manquant, ignoré")
        results[table_name] = None
        continue

    t0 = time.time()
    encoding = 'latin-1' if table_name == 'raw_fao_historique' else 'utf-8'

    try:
        # Lire avec pandas
        df = pd.read_csv(csv_path, encoding=encoding, low_memory=False)

        # Normaliser les noms de colonnes pour PostgreSQL
        # (espaces, parenthèses et caractères spéciaux → underscores)
        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(r'[^a-z0-9]+', '_', regex=True)
            .str.strip('_')
        )

        # Créer la table et insérer
        df.to_sql(table_name, engine, schema='raw',
                  if_exists='replace', index=False,
                  method='multi', chunksize=5000)

        n_rows = df.shape[0]
        elapsed = time.time() - t0
        results[table_name] = n_rows
        print(f"  ✅ raw.{table_name:<30} {n_rows:>9,} lignes × {len(df.columns)} cols  ({elapsed:.1f}s)")

    except Exception as e:
        print(f"  ❌ raw.{table_name} — ERREUR : {e}")
        results[table_name] = None

total = sum(v for v in results.values() if v)
print(f"\n✅ Ingestion terminée — {total:,} lignes au total")

In [ ]:
# 2.2.3 — Validation des tables raw
print("🔍 Aperçu des tables raw ingérées :")
print("=" * 65)

tables_raw = query_df("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
""")

for _, row in tables_raw.iterrows():
    t    = row['table_name']
    full = f"raw.{t}"
    n    = query_df(f"SELECT COUNT(*) as n FROM {full}")['n'].iloc[0]
    cols = query_df(f"""
        SELECT column_name FROM information_schema.columns
        WHERE table_schema = 'raw' AND table_name = '{t}'
        ORDER BY ordinal_position
    """)['column_name'].tolist()
    print(f"\n  📁 {full}")
    print(f"     Lignes   : {n:,}")
    print(f"     Colonnes : {cols}")

In [ ]:
# 2.2.4 — Tests SQL de validation (cross-schema raw ↔ public)
print("🔌 Tests SQL")
print("=" * 60)

# Test 1 : Top 5 pays producteurs de blé 2020 (raw)
if results.get('raw_fao_complet'):
    print("\n📊 Test 1 — Top 5 pays producteurs de blé, 2020 :")
    df1 = query_df("""
        SELECT area, SUM(value) as total_1000t
        FROM raw.raw_fao_complet
        WHERE item ILIKE '%Wheat%'
          AND element = 'Food'
          AND year = 2020
        GROUP BY area
        ORDER BY total_1000t DESC
        LIMIT 5
    """)
    print(df1.to_string(index=False))

# Test 2 : Top 5 produits les plus carbonés (raw)
if results.get('raw_food_production'):
    print("\n📊 Test 2 — Top 5 produits les plus carbonés :")
    cols_fp = query_df("""
        SELECT column_name FROM information_schema.columns
        WHERE table_schema = 'raw' AND table_name = 'raw_food_production'
        ORDER BY ordinal_position
    """)['column_name'].tolist()
    col_produit = cols_fp[0]
    col_co2     = next((c for c in cols_fp if 'total_emission' in c), cols_fp[7])
    df2 = query_df(f"""
        SELECT "{col_produit}", "{col_co2}" as co2_per_kg
        FROM raw.raw_food_production
        WHERE "{col_co2}" IS NOT NULL
        ORDER BY "{col_co2}" DESC
        LIMIT 5
    """)
    print(df2.to_string(index=False))

# Test 3 : Toutes les tables par schéma
print("\n📊 Test 3 — Inventaire complet :")
all_tables = query_df("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema IN ('public', 'raw')
    ORDER BY table_schema, table_name
""")
for schema, group in all_tables.groupby('table_schema'):
    label = 'Data Lake' if schema == 'raw' else 'Data Warehouse'
    print(f"  [{label} — {schema}] {group['table_name'].tolist()}")

print("\n✅ PostgreSQL opérationnel — raw + public accessibles")

In [ ]:
# 2.2.5 — Résumé
n_ok  = sum(1 for v in results.values() if v is not None)
n_ko  = sum(1 for v in results.values() if v is None)
total = sum(v for v in results.values() if v)

print("=" * 60)
print("✅ DATA LAKE & SCHÉMA DWH CRÉÉS")
print("=" * 60)
print(f"""
Base        : PostgreSQL {PG_HOST}:{PG_PORT}/{PG_DB}
Schéma raw  : {n_ok} tables ingérées, {n_ko} manquantes
Lignes raw  : {total:,} au total
Schéma DWH  : 7 tables vides — prêtes pour ETL

Migration Scaleway (quand prêt) :
  1. Mettre à jour ../.env avec les credentials Scaleway
  2. Relancer ce notebook (engine se reconnecte automatiquement)
  → Aucun changement de code nécessaire
""")

---
## 2.3 — Dimension Pays

In [ ]:
# 2.3 — dim_pays : pays uniques avec codes ISO depuis pycountry
import pycountry

# Corrections manuelles FAO → ISO3 (noms non-standards FAO)
FAO_TO_ISO3 = {
    'China, mainland'                                    : 'CHN',
    'China, Taiwan Province of'                          : 'TWN',
    'China, Hong Kong SAR'                               : 'HKG',
    'China, Macao SAR'                                   : 'MAC',
    'Bolivia (Plurinational State of)'                   : 'BOL',
    'Venezuela (Bolivarian Republic of)'                 : 'VEN',
    'United States of America'                           : 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'GBR',
    'Republic of Korea'                                  : 'KOR',
    "Democratic People's Republic of Korea"              : 'PRK',
    'Viet Nam'                                           : 'VNM',
    'Iran (Islamic Republic of)'                         : 'IRN',
    'Syrian Arab Republic'                               : 'SYR',
    "Lao People's Democratic Republic"                   : 'LAO',
    "Côte d'Ivoire"                                      : 'CIV',
    'Czech Republic'                                     : 'CZE',
    'Czechia'                                            : 'CZE',
    'Republic of Moldova'                                : 'MDA',
    'Russian Federation'                                 : 'RUS',
    'United Republic of Tanzania'                        : 'TZA',
    'United Arab Emirates'                               : 'ARE',
    'Papua New Guinea'                                   : 'PNG',
    'Congo'                                              : 'COG',
    'Democratic Republic of the Congo'                   : 'COD',
    'Eswatini'                                           : 'SWZ',
    'North Macedonia'                                    : 'MKD',
    'The former Yugoslav Republic of Macedonia'          : 'MKD',
    'Occupied Palestinian Territory'                     : 'PSE',
    'Palestine'                                          : 'PSE',
    'Timor-Leste'                                        : 'TLS',
    'Micronesia (Federated States of)'                   : 'FSM',
    'Cabo Verde'                                         : 'CPV',
    'Cape Verde'                                         : 'CPV',
    'Sao Tome and Principe'                              : 'STP',
    'Sudan (former)'                                     : 'SDN',
    # Régions agrégées FAO → pas de pays ISO (exclus du DWH)
    'Africa': None, 'Americas': None, 'Asia': None, 'Europe': None,
    'Oceania': None, 'World': None,
    'Low Income Food Deficit Countries'    : None,
    'Net Food Importing Developing Countries': None,
    'Least Developed Countries'            : None,
    'Land Locked Developing Countries'     : None,
    'Small Island Developing States'       : None,
    'Eastern Africa': None, 'Middle Africa': None, 'Northern Africa': None,
    'Southern Africa': None, 'Western Africa': None,
    'Central Asia': None, 'Eastern Asia': None, 'South-eastern Asia': None,
    'Southern Asia': None, 'Western Asia': None,
    'Eastern Europe': None, 'Northern Europe': None,
    'Southern Europe': None, 'Western Europe': None,
    'Caribbean': None, 'Central America': None, 'South America': None,
    'Northern America': None, 'Australia and New Zealand': None,
    'Melanesia': None, 'Micronesia': None, 'Polynesia': None,
    'Serbia and Montenegro': None,
}

# Mapping ISO2 → région géographique simplifiée
REGION_MAP = {
    'AF':'Africa','AO':'Africa','BF':'Africa','BI':'Africa','BJ':'Africa',
    'BW':'Africa','CD':'Africa','CF':'Africa','CG':'Africa','CI':'Africa',
    'CM':'Africa','CV':'Africa','DJ':'Africa','DZ':'Africa','EG':'Africa',
    'ER':'Africa','ET':'Africa','GA':'Africa','GH':'Africa','GM':'Africa',
    'GN':'Africa','GQ':'Africa','GW':'Africa','KE':'Africa','KM':'Africa',
    'LR':'Africa','LS':'Africa','LY':'Africa','MA':'Africa','MG':'Africa',
    'ML':'Africa','MR':'Africa','MU':'Africa','MW':'Africa','MZ':'Africa',
    'NA':'Africa','NE':'Africa','NG':'Africa','RW':'Africa','SC':'Africa',
    'SD':'Africa','SL':'Africa','SN':'Africa','SO':'Africa','SS':'Africa',
    'ST':'Africa','SZ':'Africa','TD':'Africa','TG':'Africa','TN':'Africa',
    'TZ':'Africa','UG':'Africa','ZA':'Africa','ZM':'Africa','ZW':'Africa',
    'AM':'Asia','AZ':'Asia','BD':'Asia','BH':'Asia','BN':'Asia','BT':'Asia',
    'CN':'Asia','CY':'Asia','GE':'Asia','HK':'Asia','ID':'Asia','IL':'Asia',
    'IN':'Asia','IQ':'Asia','IR':'Asia','JO':'Asia','JP':'Asia','KG':'Asia',
    'KH':'Asia','KP':'Asia','KR':'Asia','KW':'Asia','KZ':'Asia','LA':'Asia',
    'LB':'Asia','LK':'Asia','MM':'Asia','MN':'Asia','MO':'Asia','MV':'Asia',
    'MY':'Asia','NP':'Asia','OM':'Asia','PH':'Asia','PK':'Asia','PS':'Asia',
    'QA':'Asia','SA':'Asia','SG':'Asia','SY':'Asia','TH':'Asia','TJ':'Asia',
    'TL':'Asia','TM':'Asia','TR':'Asia','TW':'Asia','UZ':'Asia','VN':'Asia',
    'YE':'Asia',
    'AL':'Europe','AT':'Europe','BA':'Europe','BE':'Europe','BG':'Europe',
    'BY':'Europe','CH':'Europe','CZ':'Europe','DE':'Europe','DK':'Europe',
    'EE':'Europe','ES':'Europe','FI':'Europe','FR':'Europe','GB':'Europe',
    'GR':'Europe','HR':'Europe','HU':'Europe','IE':'Europe','IS':'Europe',
    'IT':'Europe','LI':'Europe','LT':'Europe','LU':'Europe','LV':'Europe',
    'MD':'Europe','ME':'Europe','MK':'Europe','MT':'Europe','NL':'Europe',
    'NO':'Europe','PL':'Europe','PT':'Europe','RO':'Europe','RS':'Europe',
    'RU':'Europe','SE':'Europe','SI':'Europe','SK':'Europe','SM':'Europe',
    'UA':'Europe',
    'AG':'Americas','AR':'Americas','BB':'Americas','BO':'Americas',
    'BR':'Americas','BS':'Americas','BZ':'Americas','CA':'Americas',
    'CL':'Americas','CO':'Americas','CR':'Americas','CU':'Americas',
    'DO':'Americas','EC':'Americas','GT':'Americas','GY':'Americas',
    'HN':'Americas','HT':'Americas','JM':'Americas','KN':'Americas',
    'LC':'Americas','MX':'Americas','NI':'Americas','PA':'Americas',
    'PE':'Americas','PY':'Americas','SR':'Americas','SV':'Americas',
    'TT':'Americas','US':'Americas','UY':'Americas','VC':'Americas',
    'VE':'Americas',
    'AU':'Oceania','FJ':'Oceania','KI':'Oceania','NR':'Oceania',
    'NZ':'Oceania','PG':'Oceania','PW':'Oceania','SB':'Oceania',
    'TO':'Oceania','TV':'Oceania','VU':'Oceania','WS':'Oceania',
}

def get_iso3(name):
    if name in FAO_TO_ISO3:
        return FAO_TO_ISO3[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        pass
    try:
        matches = pycountry.countries.search_fuzzy(name)
        if matches:
            return matches[0].alpha_3
    except Exception:
        pass
    return None

# Extraire pays uniques depuis raw_fao_complet
print("📥 Extraction des pays FAO...")
areas = query_df("SELECT DISTINCT area FROM raw.raw_fao_complet ORDER BY area")['area'].tolist()
print(f"   {len(areas)} entrées uniques")

rows, unmapped = [], []
for area in areas:
    iso3 = get_iso3(area)
    if iso3 is None:
        unmapped.append(area)
        continue  # on exclut les régions agrégées

    c    = pycountry.countries.get(alpha_3=iso3)
    iso2 = c.alpha_2 if c else None
    rows.append({
        'nom_pays' : area,
        'code_fao' : None,
        'code_iso2': iso2,
        'code_iso3': iso3,
        'region'   : REGION_MAP.get(iso2),
        'latitude' : None,
        'longitude': None,
    })

df_pays = pd.DataFrame(rows)
ingest_df(df_pays, 'public', 'dim_pays')

print(f"\n✅ dim_pays : {len(df_pays)} pays insérés")
print(f"   Avec région  : {df_pays['region'].notna().sum()}")
print(f"   Exclus (régions FAO) : {len(unmapped)}")
if unmapped:
    print(f"   → {unmapped[:5]}{'...' if len(unmapped)>5 else ''}")


---
## 2.4 — Dimension Temps

In [ ]:
# 2.4 — dim_temps : 1961-2023
annees = list(range(1961, 2024))
df_temps = pd.DataFrame({
    'annee'   : annees,
    'decennie': [(a // 10) * 10 for a in annees],
    'periode' : [f"{(a//10)*10}s" for a in annees],
})
ingest_df(df_temps, 'public', 'dim_temps')
print(f"✅ dim_temps : {len(df_temps)} années insérées (1961–2023)")
print(df_temps.sample(5).sort_values('annee').to_string(index=False))


---
## 2.5 — Dimension Produits

In [ ]:
# 2.5 — dim_produits : items FAO enrichis + matching Food_Production

try:
    from rapidfuzz import process as rfprocess, fuzz
    USE_RAPIDFUZZ = True
except ImportError:
    from difflib import get_close_matches
    USE_RAPIDFUZZ = False

# ── Produits Food_Production (colonne 1 = nom produit) ──
fp_cols = query_df("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'raw_food_production'
    ORDER BY ordinal_position
""")['column_name'].tolist()
col_fp_produit = fp_cols[0]
fp_names = query_df(
    f'SELECT DISTINCT "{col_fp_produit}" FROM raw.raw_food_production'
)[col_fp_produit].dropna().tolist()
print(f"   {len(fp_names)} produits Food_Production")

# ── Items FAO ──
fao_items = query_df(
    "SELECT DISTINCT item FROM raw.raw_fao_complet ORDER BY item"
)['item'].tolist()
print(f"   {len(fao_items)} items FAO")

# ── Catégorisation ──
def categorize(name):
    n = name.lower()
    if any(w in n for w in ['bovine','beef','mutton','goat','pig','pork','poultry','chicken','lamb','meat','offal']):
        sub = 'Bovine' if 'bovine' in n or 'beef' in n else               'Sheep/Goat' if 'mutton' in n or 'goat' in n else               'Pork' if 'pig' in n or 'pork' in n else               'Poultry' if 'poultry' in n or 'chicken' in n else 'Other'
        return 'Meat', sub
    if any(w in n for w in ['milk','cheese','butter','dairy','ghee','cream','whey','lactose']):
        return 'Dairy', 'Other'
    if any(w in n for w in ['egg']):
        return 'Eggs', 'Eggs'
    if any(w in n for w in ['fish','seafood','shrimp','salmon','tuna','crab','aquatic','mollusc']):
        return 'Fish & Seafood', 'Other'
    if any(w in n for w in ['wheat','rye','barley','oat','millet','sorghum','cereal','maize','corn','rice','grain','buckwheat']):
        return 'Cereals', 'Other'
    if any(w in n for w in ['soy','pea','bean','lentil','pulse','legume','groundnut','chickpea','cowpea','pigeon']):
        return 'Legumes', 'Other'
    if any(w in n for w in ['nut','cashew','almond','walnut','hazelnut','pistachio']):
        return 'Nuts', 'Other'
    if any(w in n for w in ['palm','sunflower','rapeseed','olive','soybean oil','vegetable oil','oilcrop','cotton','linseed','sesame']):
        return 'Oils & Fats', 'Other'
    if any(w in n for w in ['sugar cane','sugar beet','sugar','sweetener','fructose']):
        return 'Sugar', 'Other'
    if any(w in n for w in ['potato','cassava','yam','taro','tuber','root','sweet potato']):
        return 'Roots & Tubers', 'Other'
    if any(w in n for w in ['tomato','onion','carrot','cabbage','pepper','lettuce','spinach','vegetable','cucumber','garlic','leek']):
        return 'Vegetables', 'Other'
    if any(w in n for w in ['apple','orange','banana','grape','mango','pineapple','citrus','fruit','berr','melon','pear','avocado']):
        return 'Fruits', 'Other'
    if any(w in n for w in ['coffee','tea','cocoa','cacao','mate']):
        return 'Beverages & Stimulants', 'Other'
    if any(w in n for w in ['wine','beer','alcohol','spirit','fermented']):
        return 'Alcohol', 'Other'
    if any(w in n for w in ['spice','pepper','ginger','cinnamon','vanilla','herb']):
        return 'Spices', 'Other'
    return 'Other', None

def fuzzy_match(name, candidates):
    if USE_RAPIDFUZZ:
        res = rfprocess.extractOne(name, candidates, scorer=fuzz.token_sort_ratio)
        if res and res[1] >= 55:
            return res[0], int(res[1])
        return None, 0
    else:
        m = get_close_matches(name.lower(), [c.lower() for c in candidates], n=1, cutoff=0.45)
        if m:
            idx = [c.lower() for c in candidates].index(m[0])
            return candidates[idx], 65
        return None, 0

rows_prod = []
for item in fao_items:
    cat, sub = categorize(item)
    nom_impact, score = fuzzy_match(item, fp_names)
    quality = None
    if nom_impact:
        quality = 'high' if score >= 80 else 'medium' if score >= 55 else 'low'
        if quality == 'low':
            nom_impact, quality = None, None
    rows_prod.append({
        'nom_fao'       : item,
        'nom_impact'    : nom_impact,
        'categorie'     : cat,
        'sous_categorie': sub,
        'match_quality' : quality,
    })

df_produits = pd.DataFrame(rows_prod)
ingest_df(df_produits, 'public', 'dim_produits')
print(f"\n✅ dim_produits : {len(df_produits)} produits insérés")
print(f"   Matchés Food_Production  : {df_produits['nom_impact'].notna().sum()}")
print(f"   Match high  : {(df_produits['match_quality']=='high').sum()}")
print(f"   Match medium: {(df_produits['match_quality']=='medium').sum()}")
print("\n📋 Exemples de matchs :")
sample = df_produits[df_produits['nom_impact'].notna()][['nom_fao','nom_impact','match_quality','categorie']].head(10)
print(sample.to_string(index=False))


In [ ]:
# Validation des 4 dimensions
print("=" * 60)
print("📋 Dimensions DWH — résumé")
print("=" * 60)
for tbl in ['dim_pays', 'dim_temps', 'dim_produits']:
    n = query_df(f"SELECT COUNT(*) as n FROM public.{tbl}")['n'].iloc[0]
    print(f"  {tbl:<28} {n:>6} lignes")

# Aperçu par région
print("\n  dim_pays — répartition par région :")
reg = query_df("""
    SELECT COALESCE(region, 'Non mappé') as region, COUNT(*) as n
    FROM public.dim_pays GROUP BY region ORDER BY n DESC
""")
print(reg.to_string(index=False))
print("\n  dim_produits — répartition par catégorie :")
cat = query_df("""
    SELECT categorie, COUNT(*) as n
    FROM public.dim_produits GROUP BY categorie ORDER BY n DESC
""")
print(cat.to_string(index=False))


---
## 2.6 — Dimension Socio-économique (World Bank)

In [ ]:
# 2.6 — dim_socio_economique : données World Bank × dim_pays × dim_temps

wb_all_cols = query_df("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'raw_worldbank'
    ORDER BY ordinal_position
""")['column_name'].tolist()
print("Colonnes raw_worldbank :", wb_all_cols)

def find_col(cols, candidates):
    for cand in candidates:
        matches = [c for c in cols if cand.lower() in c.lower()]
        if matches:
            return matches[0]
    return None

col_iso3 = find_col(wb_all_cols, ['iso3','alpha_3','iso_3','code_iso3','code'])
col_year = find_col(wb_all_cols, ['year','annee'])
col_gdp  = find_col(wb_all_cols, ['gdp','pib','ny_gdp'])
col_urb  = find_col(wb_all_cols, ['urb','urban'])
col_pop  = find_col(wb_all_cols, ['pop','population','sp_pop'])
col_agr  = find_col(wb_all_cols, ['agri','ag_lnd','land'])
print(f"→ iso3={col_iso3}, year={col_year}, gdp={col_gdp}, urb={col_urb}, pop={col_pop}, agri={col_agr}")

df_wb = query_df("SELECT * FROM raw.raw_worldbank")

# Renommer → noms standardisés
rename = {}
if col_iso3: rename[col_iso3] = 'code_iso3'
if col_year: rename[col_year] = 'annee'
if col_gdp : rename[col_gdp]  = 'pib_per_capita'
if col_urb : rename[col_urb]  = 'taux_urbanisation'
if col_pop : rename[col_pop]  = 'population'
if col_agr : rename[col_agr]  = 'surface_agricole'
df_wb = df_wb.rename(columns=rename)

# Convertir l'année en int
df_wb['annee'] = pd.to_numeric(df_wb['annee'], errors='coerce').astype('Int64')

# Charger les FK
df_dim_pays  = query_df("SELECT pays_id, code_iso3 FROM public.dim_pays WHERE code_iso3 IS NOT NULL")
df_dim_temps = query_df("SELECT annee_id, annee    FROM public.dim_temps")
df_dim_temps['annee'] = df_dim_temps['annee'].astype('Int64')

# Joindre sur iso3 + année
cols_keep = ['pays_id','annee_id']
num_cols  = ['pib_per_capita','taux_urbanisation','population','surface_agricole']
for c in num_cols:
    if c not in df_wb.columns:
        df_wb[c] = np.nan

df_socio = (
    df_wb
    .merge(df_dim_pays,  on='code_iso3', how='inner')
    .merge(df_dim_temps, on='annee',     how='inner')
)[cols_keep + num_cols]

# Conversions de types
for c in ['pib_per_capita','taux_urbanisation','surface_agricole']:
    df_socio[c] = pd.to_numeric(df_socio[c], errors='coerce')
df_socio['population'] = pd.to_numeric(df_socio['population'], errors='coerce').astype('Int64')

df_socio = df_socio.drop_duplicates(subset=['pays_id','annee_id'])

ingest_df(df_socio, 'public', 'dim_socio_economique')
print(f"\n✅ dim_socio_economique : {len(df_socio):,} lignes insérées")
print(f"   Pays couverts  : {df_socio['pays_id'].nunique()}")
print(f"   Années couvertes: {df_socio['annee_id'].nunique()}")
print(f"   PIB non-null   : {df_socio['pib_per_capita'].notna().sum():,}")
print(f"   Pop non-null   : {df_socio['population'].notna().sum():,}")


---
## 2.7 — Faits : Production & Facteurs d'Impact Environnemental

In [ ]:
# 2.7.1 — fait_impact : facteurs CO2/eau/land par produit (Food_Production)
# Idempotent : on vide la table avant de la repeupler
run_sql("TRUNCATE TABLE public.fait_impact CASCADE")

fp_cols = query_df("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'raw_food_production'
    ORDER BY ordinal_position
""")['column_name'].tolist()

col_fp_produit = fp_cols[0]

def find_fp_col(cols, candidates):
    for cand in candidates:
        m = [c for c in cols if cand.lower() in c.lower()]
        if m:
            return m[0]
    return None

col_land_chg = find_fp_col(fp_cols, ['land_use_change'])
col_feed     = find_fp_col(fp_cols, ['animal_feed'])
col_farm     = find_fp_col(fp_cols, ['farm'])
col_proc     = find_fp_col(fp_cols, ['processing'])
col_transp   = find_fp_col(fp_cols, ['transport'])
col_pack     = find_fp_col(fp_cols, ['packging', 'packaging'])
col_retail   = find_fp_col(fp_cols, ['retail'])
col_co2      = find_fp_col(fp_cols, ['total_emission', 'total_ghg'])
# ✅ Recherche explicite sur "per_kilogram" pour éviter les colonnes per_1000kcal
col_water    = find_fp_col(fp_cols, ['freshwater_withdrawals_per_kilogram'])
col_scarc    = find_fp_col(fp_cols, ['scarcity_weighted_water_use_per_kilogram'])
col_land     = find_fp_col(fp_cols, ['land_use_per_kilogram'])
col_eutro    = find_fp_col(fp_cols, ['eutrophying_emissions_per_kilogram'])

print("Colonnes détectées :")
for lbl, c in [('total_co2', col_co2), ('land_change', col_land_chg), ('farm', col_farm),
               ('water/kg',  col_water), ('land/kg', col_land), ('eutrophying/kg', col_eutro),
               ('scarcity/kg', col_scarc)]:
    print(f"  {lbl:<15} → {c}")

df_fp = query_df("SELECT * FROM raw.raw_food_production")
df_dim_prod = query_df(
    "SELECT produit_id, nom_impact FROM public.dim_produits WHERE nom_impact IS NOT NULL"
)

# Normaliser clé de join
df_fp['_key']       = df_fp[col_fp_produit].str.strip()
df_dim_prod['_key'] = df_dim_prod['nom_impact'].str.strip()
df_fi = df_fp.merge(df_dim_prod, on='_key', how='inner')

col_map = {
    'produit_id' : 'produit_id',
    col_land_chg : 'co2_land_use_per_kg',
    col_feed     : 'co2_animal_feed_per_kg',
    col_farm     : 'co2_farm_per_kg',
    col_proc     : 'co2_processing_per_kg',
    col_transp   : 'co2_transport_per_kg',
    col_pack     : 'co2_packaging_per_kg',
    col_retail   : 'co2_retail_per_kg',
    col_co2      : 'co2_total_per_kg',
    col_water    : 'freshwater_per_kg',
    col_scarc    : 'scarcity_water_per_kg',
    col_land     : 'land_use_per_kg',
    col_eutro    : 'eutrophying_per_kg',
}
col_map_clean = {k: v for k, v in col_map.items() if k}

df_fi_dwh = df_fi[list(col_map_clean.keys())].rename(columns=col_map_clean)
for c in df_fi_dwh.columns:
    if c != 'produit_id':
        df_fi_dwh[c] = pd.to_numeric(df_fi_dwh[c], errors='coerce')

ingest_df(df_fi_dwh, 'public', 'fait_impact')
print(f"\n✅ fait_impact : {len(df_fi_dwh)} produits enrichis avec facteurs environnementaux")
print(f"   CO2 total non-null   : {df_fi_dwh['co2_total_per_kg'].notna().sum()}")
print(f"   Eau/kg non-null      : {df_fi_dwh['freshwater_per_kg'].notna().sum()}")
print(f"   Land/kg non-null     : {df_fi_dwh['land_use_per_kg'].notna().sum()}")
print(f"\n📋 Top 5 produits les plus carbonés (kgCO2/kg) :")
print(df_fi_dwh.nlargest(5, 'co2_total_per_kg')[['produit_id','co2_total_per_kg','freshwater_per_kg','land_use_per_kg']].to_string(index=False))

In [ ]:
# 2.7.2 — fait_production : quantités FAO (Food + Feed) → star schema
# Idempotent : on vide la table avant de la repeupler
run_sql("TRUNCATE TABLE public.fait_production CASCADE")

print("📥 Chargement raw_fao_complet (Food + Feed uniquement)...")
df_fao = query_df("""
    SELECT area, item, element, year::INTEGER as year, value
    FROM raw.raw_fao_complet
    WHERE element IN ('Food', 'Feed')
      AND value IS NOT NULL
""")
print(f"   {len(df_fao):,} lignes chargées")

# Charger les FK
df_dp  = query_df("SELECT pays_id, nom_pays FROM public.dim_pays")
df_dpr = query_df("SELECT produit_id, nom_fao FROM public.dim_produits")
df_dt  = query_df("SELECT annee_id, annee FROM public.dim_temps")
df_dt['annee'] = df_dt['annee'].astype(int)
df_fao['year'] = df_fao['year'].astype(int)

# Jointures
df_prod = (
    df_fao
    .merge(df_dp,  left_on='area', right_on='nom_pays',  how='inner')
    .merge(df_dpr, left_on='item', right_on='nom_fao',   how='inner')
    .merge(df_dt,  left_on='year', right_on='annee',     how='inner')
)[['pays_id', 'produit_id', 'annee_id', 'element', 'value']]

df_prod = df_prod.rename(columns={'value': 'quantite_1000t'})
df_prod['element']        = df_prod['element'].str[:10]
df_prod['quantite_1000t'] = pd.to_numeric(df_prod['quantite_1000t'], errors='coerce')
df_prod = df_prod.drop_duplicates(subset=['pays_id', 'produit_id', 'annee_id', 'element'])

print(f"   Après join  : {len(df_prod):,} lignes")
print(f"   Pays        : {df_prod['pays_id'].nunique()}")
print(f"   Produits    : {df_prod['produit_id'].nunique()}")
print(f"   Années      : {df_prod['annee_id'].nunique()}")

ingest_df(df_prod, 'public', 'fait_production')
print(f"\n✅ fait_production : {len(df_prod):,} lignes insérées")

---
## 2.8 — Table Principale ML : `fait_impact_pays_annee`

Table dénormalisée optimisée pour le ML :
- 1 ligne = 1 pays × 1 année × 1 produit (Food uniquement)
- Quantités FAO × facteurs Food_Production → CO2 / eau / land
- `co2_total_kg` = NULL si le produit n'a pas de match Food_Production

In [ ]:
# 2.8 — fait_impact_pays_annee : jointure production × impact → table ML
# Idempotent : on vide la table avant de la repeupler
run_sql("TRUNCATE TABLE public.fait_impact_pays_annee")

print("📥 Construction fait_impact_pays_annee...")

# Agréger fait_production sur Food uniquement, par (pays, produit, année)
df_fprod = query_df("""
    SELECT pays_id, annee_id, produit_id,
           SUM(quantite_1000t) as quantite_1000t
    FROM public.fait_production
    WHERE element = 'Food'
    GROUP BY pays_id, annee_id, produit_id
""")
print(f"   fait_production (Food) : {len(df_fprod):,} lignes")

# Charger les facteurs d'impact
df_fimpact = query_df("""
    SELECT produit_id, co2_total_per_kg, freshwater_per_kg,
           land_use_per_kg, eutrophying_per_kg
    FROM public.fait_impact
""")
print(f"   fait_impact            : {len(df_fimpact)} produits avec facteurs")

# Jointure LEFT pour garder tous les produits FAO (co2 = NULL si pas de match)
df_main = df_fprod.merge(df_fimpact, on='produit_id', how='left')

# 1000 t → kg  (×1 000 000)
df_main['quantite_kg']             = df_main['quantite_1000t'] * 1e6
df_main['co2_total_kg']            = df_main['quantite_kg'] * df_main['co2_total_per_kg']
df_main['freshwater_total_litres'] = df_main['quantite_kg'] * df_main['freshwater_per_kg']
df_main['land_use_total_m2']       = df_main['quantite_kg'] * df_main['land_use_per_kg']

df_main = df_main[[
    'pays_id', 'annee_id', 'produit_id',
    'quantite_1000t', 'quantite_kg',
    'co2_total_kg', 'freshwater_total_litres', 'land_use_total_m2',
]].drop_duplicates(subset=['pays_id', 'annee_id', 'produit_id'])

ingest_df(df_main, 'public', 'fait_impact_pays_annee')

pct_co2 = df_main['co2_total_kg'].notna().mean() * 100
print(f"\n✅ fait_impact_pays_annee : {len(df_main):,} lignes insérées")
print(f"   CO2 non-null  : {df_main['co2_total_kg'].notna().sum():,} ({pct_co2:.0f}%)")
print(f"   Pays          : {df_main['pays_id'].nunique()}")
print(f"   Années        : {df_main['annee_id'].nunique()}")
print(f"   Produits      : {df_main['produit_id'].nunique()}")

In [ ]:
# ── Validation finale du DWH ──────────────────────────────────────────────

print("=" * 65)
print("✅ DATA WAREHOUSE COMPLET — RÉSUMÉ")
print("=" * 65)

tables = query_df("""
    SELECT t.table_schema, t.table_name,
           pg_size_pretty(pg_total_relation_size(
               quote_ident(t.table_schema) || '.' || quote_ident(t.table_name)
           )) AS size
    FROM information_schema.tables t
    WHERE t.table_schema IN ('public','raw')
      AND t.table_type = 'BASE TABLE'
    ORDER BY t.table_schema DESC, t.table_name
""")

print()
for _, row in tables.iterrows():
    n = query_df(f"SELECT COUNT(*) as n FROM {row['table_schema']}.{row['table_name']}")['n'].iloc[0]
    schema_lbl = 'DWH' if row['table_schema'] == 'public' else 'RAW'
    print(f"  [{schema_lbl}] {row['table_schema']}.{row['table_name']:<35} {n:>10,} lignes  {row['size']}")

# Test métier : Top 10 pays par empreinte CO2 en 2020
# ✅ Cast ::NUMERIC requis — ROUND(double precision, n) n'existe pas en PostgreSQL
print("\n📊 Top 10 pays — émissions CO2 alimentaires totales (2020, en Mt CO2) :")
df_test = query_df("""
    SELECT p.nom_pays,
           ROUND((SUM(fi.co2_total_kg) / 1e9)::NUMERIC, 3) as co2_Mt
    FROM public.fait_impact_pays_annee fi
    JOIN public.dim_pays  p ON p.pays_id  = fi.pays_id
    JOIN public.dim_temps t ON t.annee_id = fi.annee_id
    WHERE t.annee = 2020
      AND fi.co2_total_kg IS NOT NULL
    GROUP BY p.nom_pays
    ORDER BY co2_Mt DESC
    LIMIT 10
""")
print(df_test.to_string(index=False))

print(f"""
{'─'*65}
Migration Scaleway (quand prêt) :
  1. Mettre à jour ../.env avec les credentials Scaleway
  2. Relancer ce notebook (engine se reconnecte automatiquement)
  → Aucun changement de code nécessaire
{'─'*65}""")